In [1]:
!pip install implicit --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 914.7 kB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# ============================================
# CELL 1 — IMPORTS + CONSTANTS
# ============================================
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

# Core column names
COL_USER   = "reviewerID"
COL_ITEM   = "asin"
COL_RATING = "overall"
COL_DATE   = "review_date"

CATEGORY = "tools_&_home_improvement"

# ---- INPUT ARTIFACTS ----
# đổi path nếu dataset Kaggle của bạn đang mount tên khác
STAGE0_DIR = Path("/kaggle/input/datasets/trinhdieulinh/stage0/stage0_artifacts")
STAGE2_DIR = Path("/kaggle/input/datasets/trinhdieulinh/stage2/stage2_artifacts")

# ---- OUTPUT DIR ----
STAGE3_DIR = Path("/kaggle/working/stage3_artifacts")
STAGE3_DIR.mkdir(parents=True, exist_ok=True)

print("Stage0 input :", STAGE0_DIR)
print("Stage2 input :", STAGE2_DIR)
print("Stage3 output:", STAGE3_DIR)

Stage0 input : /kaggle/input/datasets/trinhdieulinh/stage0/stage0_artifacts
Stage2 input : /kaggle/input/datasets/trinhdieulinh/stage2/stage2_artifacts
Stage3 output: /kaggle/working/stage3_artifacts


In [3]:
# ============================================
# CELL 2 — LOAD STAGE 0 / STAGE 2 ARTIFACTS
# ============================================

# ---------- Stage 0 ----------
with open(STAGE0_DIR / "split_meta.json", "r") as f:
    split_meta = json.load(f)

with open(STAGE0_DIR / "baseline_scores.json", "r") as f:
    baseline_scores = json.load(f)

train   = pd.read_parquet(STAGE0_DIR / "train.parquet").copy()
test    = pd.read_parquet(STAGE0_DIR / "test.parquet").copy()
df_core = pd.read_parquet(STAGE0_DIR / "df_core.parquet").copy()

user_means_df = pd.read_parquet(STAGE0_DIR / "user_means.parquet")
item_means_df = pd.read_parquet(STAGE0_DIR / "item_means.parquet")

# ---------- Stage 2 ----------
with open(STAGE2_DIR / "encodings.pkl", "rb") as f:
    enc = pickle.load(f)

user_enc_s2    = enc["user_enc_s2"]
item_enc_s2_cf = enc["item_enc_s2_cf"]
item_enc_s2    = enc["item_enc_s2"]

user_means_vec = enc["user_means_vec"]
item_means_vec = enc["item_means_vec"]
GLOBAL_MEAN    = float(enc["global_mean"])

M_norm        = sp.load_npz(STAGE2_DIR / "M_norm.npz")
C_implicit    = sp.load_npz(STAGE2_DIR / "C_implicit.npz")
item_profiles = np.load(STAGE2_DIR / "item_profiles.npy")
user_profiles = np.load(STAGE2_DIR / "user_profiles.npy")

# ---------- Basic cleanup ----------
if COL_DATE in train.columns:
    train[COL_DATE] = pd.to_datetime(train[COL_DATE], errors="coerce")
if COL_DATE in test.columns:
    test[COL_DATE] = pd.to_datetime(test[COL_DATE], errors="coerce")
if COL_DATE in df_core.columns:
    df_core[COL_DATE] = pd.to_datetime(df_core[COL_DATE], errors="coerce")

# ---------- Mean maps ----------
# hỗ trợ cả trường hợp reviewerID/asin đang là cột thường hoặc index
if COL_USER in user_means_df.columns:
    user_mean_map = dict(zip(user_means_df[COL_USER], user_means_df["user_mean"]))
else:
    user_mean_map = user_means_df["user_mean"].to_dict()

if COL_ITEM in item_means_df.columns:
    item_mean_map = dict(zip(item_means_df[COL_ITEM], item_means_df["item_mean"]))
else:
    item_mean_map = item_means_df["item_mean"].to_dict()

print("=" * 70)
print("Loaded Stage 0 / Stage 2 artifacts")
print("=" * 70)

print(f"train shape        : {train.shape}")
print(f"test shape         : {test.shape}")
print(f"df_core shape      : {df_core.shape}")

print(f"GLOBAL_MEAN        : {GLOBAL_MEAN:.6f}")
print(f"Baseline warm RMSE : {baseline_scores['warm_rmse']:.6f}")
print(f"Baseline warm MAE  : {baseline_scores['warm_mae']:.6f}")

print(f"M_norm shape       : {M_norm.shape}")
print(f"C_implicit shape   : {C_implicit.shape}")
print(f"item_profiles      : {item_profiles.shape}")
print(f"user_profiles      : {user_profiles.shape}")

print(f"#user_enc_s2       : {len(user_enc_s2):,}")
print(f"#item_enc_s2       : {len(item_enc_s2):,}")
print(f"#item_enc_s2_cf    : {len(item_enc_s2_cf):,}")

Loaded Stage 0 / Stage 2 artifacts
train shape        : (1131163, 4)
test shape         : (282791, 10)
df_core shape      : (1413954, 4)
GLOBAL_MEAN        : 4.381161
Baseline warm RMSE : 1.102864
Baseline warm MAE  : 0.811304
M_norm shape       : (164581, 58019)
C_implicit shape   : (164581, 58019)
item_profiles      : (58019, 128)
user_profiles      : (164581, 128)
#user_enc_s2       : 164,581
#item_enc_s2       : 58,019
#item_enc_s2_cf    : 58,019


In [4]:
# ============================================
# CELL 3 ? BUILD TEST_WARM + SANITY CHECKS + BASELINE REPLAY
# ============================================

assert len(train) == split_meta["n_train"], f"n_train mismatch: {len(train)} vs {split_meta['n_train']}"
assert len(test)  == split_meta["n_test"],  f"n_test mismatch: {len(test)} vs {split_meta['n_test']}"
assert M_norm.shape[0] == len(user_means_vec), "Mismatch: M_norm rows vs user_means_vec"
assert M_norm.shape[1] == len(item_means_vec), "Mismatch: M_norm cols vs item_means_vec"
assert item_profiles.shape[0] == len(item_enc_s2), "Mismatch: item_profiles rows vs item_enc_s2"
assert user_profiles.shape[0] == len(user_enc_s2), "Mismatch: user_profiles rows vs user_enc_s2"

train_users = set(train[COL_USER].unique())
train_items = set(train[COL_ITEM].unique())

test = test.copy()
test["user_known"] = test[COL_USER].isin(train_users)
test["item_known"] = test[COL_ITEM].isin(train_items)

mask_warm    = test["user_known"] & test["item_known"]
mask_cold_u  = ~test["user_known"] & test["item_known"]
mask_cold_i  =  test["user_known"] & ~test["item_known"]
mask_cold_ui = ~test["user_known"] & ~test["item_known"]

test_warm = test.loc[mask_warm].reset_index(drop=True)

def clip_rating(x):
    return np.clip(np.asarray(x, dtype=np.float32), 1.0, 5.0)

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    return float(np.mean(np.abs(y_true - y_pred)))

def evaluate_predictions(y_true, y_pred, name="model"):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = clip_rating(y_pred)
    out = {"model": name, "n": int(len(y_true)), "rmse": rmse(y_true, y_pred), "mae": mae(y_true, y_pred), "per_star": {}}
    for s in [1, 2, 3, 4, 5]:
        mask = (y_true == s)
        if mask.sum() > 0:
            out["per_star"][f"star_{s}_rmse"] = rmse(y_true[mask], y_pred[mask])
            out["per_star"][f"star_{s}_mae"] = mae(y_true[mask], y_pred[mask])
            out["per_star"][f"star_{s}_n"] = int(mask.sum())
    return out

def print_eval(res):
    print("=" * 60)
    print("Model:", res["model"])
    print("N    :", res["n"])
    print("RMSE :", round(res["rmse"], 6))
    print("MAE  :", round(res["mae"], 6))
    print("-" * 60)
    for k, v in res["per_star"].items():
        print(f"{k:14s}: {v}")
    print("=" * 60)

def baseline_predict(df_subset, user_map, item_map, global_mean):
    u = df_subset[COL_USER].map(user_map).fillna(global_mean)
    i = df_subset[COL_ITEM].map(item_map).fillna(global_mean)
    return ((u + i) / 2.0).clip(1.0, 5.0).values.astype(np.float32)

def to_jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj

def date_range_str(df_subset):
    if COL_DATE not in df_subset.columns or len(df_subset) == 0:
        return "n/a"
    valid_dates = df_subset[COL_DATE].dropna()
    if len(valid_dates) == 0:
        return "all NaT"
    return f"{valid_dates.min()} -> {valid_dates.max()}"

y_true_warm = test_warm[COL_RATING].values.astype(np.float32)
y_pred_base = baseline_predict(test_warm, user_mean_map, item_mean_map, GLOBAL_MEAN)
result_baseline = evaluate_predictions(y_true_warm, y_pred_base, "baseline_replay")
print_eval(result_baseline)

train_sorted = train.sort_values(COL_DATE, na_position="last").reset_index(drop=True)
split_idx_fit = int(len(train_sorted) * 0.90)
assert 0 < split_idx_fit < len(train_sorted), "Temporal split for train_fit/val_fit is invalid."
train_fit = train_sorted.iloc[:split_idx_fit].copy().reset_index(drop=True)
val_fit = train_sorted.iloc[split_idx_fit:].copy().reset_index(drop=True)

user_mean_map_fit = train_fit.groupby(COL_USER)[COL_RATING].mean().astype(np.float32).to_dict()
item_mean_map_fit = train_fit.groupby(COL_ITEM)[COL_RATING].mean().astype(np.float32).to_dict()
GLOBAL_MEAN_FIT = float(train_fit[COL_RATING].mean())

y_val_true_oof = val_fit[COL_RATING].astype(np.float32).values
baseline_val_preds_oof = baseline_predict(val_fit, user_mean_map_fit, item_mean_map_fit, GLOBAL_MEAN_FIT)
result_val_baseline_oof = evaluate_predictions(y_val_true_oof, baseline_val_preds_oof, "baseline_val_oof")

validation_index_cols = [COL_USER, COL_ITEM, COL_RATING]
if COL_DATE in val_fit.columns:
    validation_index_cols.append(COL_DATE)
if "user_group" in val_fit.columns:
    validation_index_cols.append("user_group")
validation_index_oof_df = val_fit.loc[:, validation_index_cols].copy()
validation_index_oof_df.insert(0, "oof_row_id", np.arange(len(validation_index_oof_df), dtype=np.int32))

print("Split summary")
print("-" * 60)
print("Expected split_date :", split_meta.get("split_date"))
print("Train               :", len(train))
print("Test                :", len(test))
print("Warm test           :", len(test_warm), f"({mask_warm.mean():.1%})")
print("Cold user           :", int(mask_cold_u.sum()))
print("Cold item           :", int(mask_cold_i.sum()))
print("Cold both           :", int(mask_cold_ui.sum()))
print("-" * 60)
print("Expected warm RMSE  :", baseline_scores["warm_rmse"])
print("Replay warm RMSE    :", result_baseline["rmse"])
print("Expected warm MAE   :", baseline_scores["warm_mae"])
print("Replay warm MAE     :", result_baseline["mae"])
print("-" * 60)
print("train_fit rows      :", len(train_fit))
print("val_fit rows        :", len(val_fit))
print("train_fit range     :", date_range_str(train_fit))
print("val_fit range       :", date_range_str(val_fit))
print("test_warm range     :", date_range_str(test_warm))
print("GLOBAL_MEAN_FIT     :", round(GLOBAL_MEAN_FIT, 6))
print("OOF baseline RMSE   :", round(result_val_baseline_oof["rmse"], 6))
print("OOF baseline MAE    :", round(result_val_baseline_oof["mae"], 6))

assert abs(result_baseline["rmse"] - baseline_scores["warm_rmse"]) < 0.02, "Baseline replay mismatch."
print("\nBaseline replay matched. test_warm stays final-eval only.")


Model: baseline_replay
N    : 240285
RMSE : 1.102864
MAE  : 0.811304
------------------------------------------------------------
star_1_rmse   : 3.2606801986694336
star_1_mae    : 3.2519047260284424
star_1_n      : 14055
star_2_rmse   : 2.290545701980591
star_2_mae    : 2.280728340148926
star_2_n      : 9923
star_3_rmse   : 1.3291175365447998
star_3_mae    : 1.3141452074050903
star_3_n      : 16456
star_4_rmse   : 0.40190058946609497
star_4_mae    : 0.3692122995853424
star_4_n      : 36279
star_5_rmse   : 0.5842506885528564
star_5_mae    : 0.559917688369751
star_5_n      : 163572
Split summary
------------------------------------------------------------
Expected split_date : 2017-05-06 00:00:00
Train               : 1131163
Test                : 282791
Warm test           : 240285 (85.0%)
Cold user           : 37248
Cold item           : 4757
Cold both           : 501
------------------------------------------------------------
Expected warm RMSE  : 1.1028638589291102
Replay warm RMSE

In [5]:
# ============================================
# CELL 4 ? MODEL C PREP
# ============================================

def l2_normalize_rows(X, eps=1e-12):
    X = np.asarray(X, dtype=np.float32)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms = np.maximum(norms, eps)
    return X / norms

def build_content_artifacts(train_source_df, item_profiles_base, item_encoder_base):
    item_profiles_base = l2_normalize_rows(item_profiles_base)
    train_items = [item_id for item_id in train_source_df[COL_ITEM].drop_duplicates().tolist() if item_id in item_encoder_base]
    assert len(train_items) > 0, "No train items can be mapped into item content profiles."
    item_row_idx = np.asarray([item_encoder_base[item_id] for item_id in train_items], dtype=np.int32)
    item_profiles_local = item_profiles_base[item_row_idx].astype(np.float32)
    item_enc_local = {item_id: idx for idx, item_id in enumerate(train_items)}
    train_known = train_source_df[train_source_df[COL_ITEM].isin(item_enc_local)].copy().reset_index(drop=True)
    user_list = train_known[COL_USER].drop_duplicates().tolist()
    user_enc_local = {user_id: idx for idx, user_id in enumerate(user_list)}
    rows = train_known[COL_USER].map(user_enc_local).astype(np.int32).values
    cols = train_known[COL_ITEM].map(item_enc_local).astype(np.int32).values
    vals = np.ones(len(train_known), dtype=np.float32)
    user_item_csr = sp.csr_matrix((vals, (rows, cols)), shape=(len(user_list), len(item_enc_local)), dtype=np.float32)
    user_profiles_local = user_item_csr @ item_profiles_local
    user_counts_local = np.asarray(user_item_csr.sum(axis=1)).ravel().astype(np.float32)
    user_profiles_local = user_profiles_local / np.maximum(user_counts_local[:, None], 1.0)
    user_profiles_local = l2_normalize_rows(user_profiles_local)
    return {"user_enc": user_enc_local, "item_enc": item_enc_local, "user_profiles": user_profiles_local.astype(np.float32), "item_profiles": item_profiles_local.astype(np.float32)}

def predict_content_based_with_fallback(df_subset, baseline_arr, content_artifacts, residual_scale):
    preds = baseline_arr.copy().astype(np.float32)
    sims = np.zeros(len(df_subset), dtype=np.float32)
    anchor = baseline_arr.copy().astype(np.float32)
    known_mask = df_subset[COL_USER].isin(content_artifacts["user_enc"]) & df_subset[COL_ITEM].isin(content_artifacts["item_enc"])
    if known_mask.any():
        known_df = df_subset.loc[known_mask].copy()
        u_idx = known_df[COL_USER].map(content_artifacts["user_enc"]).astype(np.int32).values
        i_idx = known_df[COL_ITEM].map(content_artifacts["item_enc"]).astype(np.int32).values
        U = content_artifacts["user_profiles"][u_idx]
        I = content_artifacts["item_profiles"][i_idx]
        sims_known = np.sum(U * I, axis=1).astype(np.float32)
        anchor_known = baseline_arr[known_mask.to_numpy()]
        preds_known = clip_rating(anchor_known + float(residual_scale) * sims_known)
        preds[known_mask.to_numpy()] = preds_known
        sims[known_mask.to_numpy()] = sims_known
        anchor[known_mask.to_numpy()] = anchor_known
    return preds.astype(np.float32), sims.astype(np.float32), anchor.astype(np.float32), known_mask

item_profiles_norm = l2_normalize_rows(item_profiles)
print("item_profiles_norm shape:", item_profiles_norm.shape)
print("Ready to rebuild content profiles from train_fit and full train only.")


item_profiles_norm shape: (58019, 128)
Ready to rebuild content profiles from train_fit and full train only.


In [6]:
# ============================================
# CELL 5 ? MODEL C TUNING ON val_fit + SAVE OOF
# ============================================

print("=" * 60)
print("MODEL C ? TUNE residual_scale ON val_fit")
print("=" * 60)

content_fit_artifacts = build_content_artifacts(train_fit, item_profiles_norm, item_enc_s2)
scale_grid = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.50, 0.75, 1.00]
results_c_grid = []

for rs in scale_grid:
    y_pred_tmp, sims_tmp, anchor_tmp, known_mask_tmp = predict_content_based_with_fallback(val_fit, baseline_val_preds_oof, content_fit_artifacts, rs)
    res_tmp = evaluate_predictions(y_val_true_oof, y_pred_tmp, name=f"model_c_val_rs_{rs}")
    results_c_grid.append({"residual_scale": float(rs), "rmse": float(res_tmp["rmse"]), "mae": float(res_tmp["mae"]), "n": int(res_tmp["n"]), "known_coverage": float(known_mask_tmp.mean()), "sim_mean": float(np.mean(sims_tmp)), "sim_std": float(np.std(sims_tmp))})
    print(f"rs={rs:<4.2f} | OOF RMSE={res_tmp['rmse']:.6f} | OOF MAE={res_tmp['mae']:.6f} | coverage={float(known_mask_tmp.mean()):.6f}")

results_c_grid_df = pd.DataFrame(results_c_grid).sort_values(["rmse", "mae", "residual_scale"], ascending=[True, True, True]).reset_index(drop=True)
display(results_c_grid_df)

best_rs = float(results_c_grid_df.iloc[0]["residual_scale"])
print("Best residual_scale on val_fit:", best_rs)
model_c_val_preds_oof, model_c_val_sims_oof, model_c_val_anchor_oof, model_c_val_known_mask = predict_content_based_with_fallback(val_fit, baseline_val_preds_oof, content_fit_artifacts, best_rs)
model_c_val_result_oof = evaluate_predictions(y_val_true_oof, model_c_val_preds_oof, name=f"model_c_val_oof_rs_{best_rs}")
print_eval(model_c_val_result_oof)

np.save(STAGE3_DIR / "model_c_val_preds_oof.npy", model_c_val_preds_oof)
np.save(STAGE3_DIR / "model_c_val_sims_oof.npy", model_c_val_sims_oof)
results_c_grid_df.to_csv(STAGE3_DIR / "model_c_tuning.csv", index=False)


MODEL C ? TUNE residual_scale ON val_fit
rs=0.00 | OOF RMSE=1.046057 | OOF MAE=0.743151 | coverage=0.844126
rs=0.05 | OOF RMSE=1.046576 | OOF MAE=0.726576 | coverage=0.844126
rs=0.10 | OOF RMSE=1.048694 | OOF MAE=0.710622 | coverage=0.844126
rs=0.15 | OOF RMSE=1.052225 | OOF MAE=0.695730 | coverage=0.844126
rs=0.20 | OOF RMSE=1.056954 | OOF MAE=0.682265 | coverage=0.844126
rs=0.25 | OOF RMSE=1.062607 | OOF MAE=0.670394 | coverage=0.844126
rs=0.50 | OOF RMSE=1.098276 | OOF MAE=0.633270 | coverage=0.844126
rs=0.75 | OOF RMSE=1.133575 | OOF MAE=0.619757 | coverage=0.844126
rs=1.00 | OOF RMSE=1.159917 | OOF MAE=0.616363 | coverage=0.844126


,residual_scale,rmse,mae,n,known_coverage,sim_mean,sim_std
0,0.00,1.046057,0.743151,113117,0.844126,0.77928,0.343223
1,0.05,1.046576,0.726576,113117,0.844126,0.77928,0.343223
2,0.10,1.048694,0.710622,113117,0.844126,0.77928,0.343223
3,0.15,1.052225,0.695730,113117,0.844126,0.77928,0.343223
4,0.20,1.056954,0.682265,113117,0.844126,0.77928,0.343223
5,0.25,1.062607,0.670394,113117,0.844126,0.77928,0.343223
6,0.50,1.098276,0.633270,113117,0.844126,0.77928,0.343223
7,0.75,1.133575,0.619757,113117,0.844126,0.77928,0.343223
8,1.00,1.159917,0.616363,113117,0.844126,0.77928,0.343223


Best residual_scale on val_fit: 0.0
Model: model_c_val_oof_rs_0.0
N    : 113117
RMSE : 1.046057
MAE  : 0.743151
------------------------------------------------------------
star_1_rmse   : 3.1409456729888916
star_1_mae    : 3.0944533348083496
star_1_n      : 5387
star_2_rmse   : 2.2216622829437256
star_2_mae    : 2.166825294494629
star_2_n      : 4077
star_3_rmse   : 1.3177480697631836
star_3_mae    : 1.2527437210083008
star_3_n      : 7411
star_4_rmse   : 0.5106449723243713
star_4_mae    : 0.4401628077030182
star_4_n      : 17070
star_5_rmse   : 0.6474452018737793
star_5_mae    : 0.5274767875671387
star_5_n      : 79172


In [7]:
# ============================================
# CELL 6 ? MODEL C REFIT ON FULL TRAIN + FINAL TEST_WARM SAVE
# ============================================

print("=" * 60)
print("MODEL C ? REFIT USER PROFILES ON FULL TRAIN")
print("=" * 60)

content_full_artifacts = build_content_artifacts(train, item_profiles_norm, item_enc_s2)
model_c_preds, model_c_sims, model_c_anchor, model_c_test_known_mask = predict_content_based_with_fallback(test_warm, y_pred_base, content_full_artifacts, best_rs)
assert len(model_c_preds) == len(test_warm), "Model C final predictions must align with test_warm."

result_c_best = evaluate_predictions(y_true_warm, model_c_preds, name=f"model_c_full_train_rs_{best_rs}")
result_base_c = evaluate_predictions(y_true_warm, y_pred_base, name="baseline_on_test_warm")
print_eval(result_base_c)
print_eval(result_c_best)

rmse_gain_c = result_base_c["rmse"] - result_c_best["rmse"]
mae_gain_c  = result_base_c["mae"] - result_c_best["mae"]
model_c_summary = {"best_residual_scale": float(best_rs), "val_fit_rmse": float(model_c_val_result_oof["rmse"]), "val_fit_mae": float(model_c_val_result_oof["mae"]), "val_fit_known_coverage": float(model_c_val_known_mask.mean()), "test_warm_rmse": float(result_c_best["rmse"]), "test_warm_mae": float(result_c_best["mae"]), "rmse_gain_vs_baseline_test_warm": float(rmse_gain_c), "mae_gain_vs_baseline_test_warm": float(mae_gain_c), "test_warm_known_coverage": float(model_c_test_known_mask.mean()), "test_warm_used_only_for_final_evaluation": True, "per_star": result_c_best["per_star"]}

np.save(STAGE3_DIR / "model_c_preds.npy", model_c_preds)
np.save(STAGE3_DIR / "model_c_sims.npy", model_c_sims)
np.save(STAGE3_DIR / "model_c_anchor.npy", model_c_anchor)
with open(STAGE3_DIR / "model_c_result.json", "w") as f:
    json.dump(to_jsonable(result_c_best), f, indent=2)
with open(STAGE3_DIR / "model_c_summary.json", "w") as f:
    json.dump(to_jsonable(model_c_summary), f, indent=2)


MODEL C ? REFIT USER PROFILES ON FULL TRAIN
Model: baseline_on_test_warm
N    : 240285
RMSE : 1.102864
MAE  : 0.811304
------------------------------------------------------------
star_1_rmse   : 3.2606801986694336
star_1_mae    : 3.2519047260284424
star_1_n      : 14055
star_2_rmse   : 2.290545701980591
star_2_mae    : 2.280728340148926
star_2_n      : 9923
star_3_rmse   : 1.3291175365447998
star_3_mae    : 1.3141452074050903
star_3_n      : 16456
star_4_rmse   : 0.40190058946609497
star_4_mae    : 0.3692122995853424
star_4_n      : 36279
star_5_rmse   : 0.5842506885528564
star_5_mae    : 0.559917688369751
star_5_n      : 163572
Model: model_c_full_train_rs_0.0
N    : 240285
RMSE : 1.102864
MAE  : 0.811304
------------------------------------------------------------
star_1_rmse   : 3.2606801986694336
star_1_mae    : 3.2519047260284424
star_1_n      : 14055
star_2_rmse   : 2.290545701980591
star_2_mae    : 2.280728340148926
star_2_n      : 9923
star_3_rmse   : 1.3291175365447998
star_3

Model C after pipeline cleanup and missing-mean fix achieved better MAE and improved low-star predictions, but it still underperformed the warm baseline in overall RMSE (1.2174 vs 1.2134). This suggests the content-based signal is useful as a fallback or ensemble component, but not strong enough as a standalone warm recommendation model. Therefore, the next priority should be Model A, which is more likely to surpass the baseline on the warm subset.

In [8]:
# ============================================
# CELL 7 ? MODEL A PREP (ITEM-ITEM CF)
# ============================================
from sklearn.neighbors import NearestNeighbors
import numpy as np
import pandas as pd

print("=" * 70)
print("MODEL A ? ITEM-ITEM CF")
print("=" * 70)

MAX_K = 20

def build_item_item_cf_artifacts(train_source_df):
    user_list = train_source_df[COL_USER].drop_duplicates().tolist()
    item_list = train_source_df[COL_ITEM].drop_duplicates().tolist()
    user_enc_local = {user_id: idx for idx, user_id in enumerate(user_list)}
    item_enc_local = {item_id: idx for idx, item_id in enumerate(item_list)}
    rows = train_source_df[COL_USER].map(user_enc_local).astype(np.int32).values
    cols = train_source_df[COL_ITEM].map(item_enc_local).astype(np.int32).values
    vals = train_source_df[COL_RATING].astype(np.float32).values
    M_raw_local = sp.csr_matrix((vals, (rows, cols)), shape=(len(user_list), len(item_list)), dtype=np.float32)
    user_counts_local = np.diff(M_raw_local.indptr).astype(np.int32)
    user_sums_local = np.asarray(M_raw_local.sum(axis=1)).ravel().astype(np.float32)
    user_means_raw_local = user_sums_local / np.maximum(user_counts_local, 1)
    M_user_centered_local = M_raw_local.copy().astype(np.float32)
    row_idx_local = np.repeat(np.arange(M_user_centered_local.shape[0], dtype=np.int32), np.diff(M_user_centered_local.indptr))
    M_user_centered_local.data -= user_means_raw_local[row_idx_local]
    B_local = M_raw_local.copy().astype(np.float32)
    B_local.data = np.ones_like(B_local.data, dtype=np.float32)
    return {"user_enc": user_enc_local, "item_enc": item_enc_local, "M_raw": M_raw_local, "M_user_centered": M_user_centered_local, "B": B_local, "user_means_raw": user_means_raw_local}

def precompute_item_neighbors(user_centered_csr, max_k=20):
    X_items_local = user_centered_csr.T.tocsr()
    n_neighbors_local = min(max_k + 1, X_items_local.shape[0])
    assert n_neighbors_local >= 2, "Model A needs at least 2 items to compute neighbors."
    nn_model_local = NearestNeighbors(n_neighbors=n_neighbors_local, metric="cosine", algorithm="brute", n_jobs=-1)
    nn_model_local.fit(X_items_local)
    distances_local, indices_local = nn_model_local.kneighbors(X_items_local, return_distance=True)
    sims_raw_local = 1.0 - distances_local
    return indices_local[:, 1:].astype(np.int32), sims_raw_local[:, 1:].astype(np.float32)

model_a_fit_artifacts = build_item_item_cf_artifacts(train_fit)
user_enc_a_fit = model_a_fit_artifacts["user_enc"]
item_enc_a_fit = model_a_fit_artifacts["item_enc"]
user_means_raw_a_fit = model_a_fit_artifacts["user_means_raw"]

print("train_fit users:", len(user_enc_a_fit))
print("train_fit items:", len(item_enc_a_fit))
print("train_fit matrix shape:", model_a_fit_artifacts["M_raw"].shape)


MODEL A ? ITEM-ITEM CF
train_fit users: 160026
train_fit items: 57261
train_fit matrix shape: (160026, 57261)


In [9]:
# ============================================
# CELL 8 ? PRECOMPUTE ITEM NEIGHBORS FOR train_fit
# ============================================

item_neighbors_idx_a_fit, item_neighbors_sim_a_fit = precompute_item_neighbors(model_a_fit_artifacts["M_user_centered"], max_k=MAX_K)

print("item_neighbors_idx_a_fit shape:", item_neighbors_idx_a_fit.shape)
print("item_neighbors_sim_a_fit shape:", item_neighbors_sim_a_fit.shape)
print("Example item 0 neighbors   :", item_neighbors_idx_a_fit[0][:10])
print("Example item 0 sims        :", item_neighbors_sim_a_fit[0][:10])


item_neighbors_idx_a_fit shape: (57261, 20)
item_neighbors_sim_a_fit shape: (57261, 20)
Example item 0 neighbors   : [39948 28885 23110 47541   345 34482  9666 32155   115 16721]
Example item 0 sims        : [0.13479912 0.12394363 0.07956308 0.06721592 0.06020403 0.01872635
 0.01826721 0.017205   0.01025987 0.00732493]


In [10]:
# ============================================
# CELL 9 ? CO-RATER COUNTS + PREDICT (FIXED)
# ============================================

def compute_neighbor_co_counts(B_csr, neighbor_idx_matrix):
    print("Computing B^T @ B (vectorized co-counts)...")
    import time
    t0 = time.time()
    B_csc = B_csr.tocsc()
    BtB = (B_csc.T @ B_csc).tocsr()
    print(f"  B^T @ B done in {time.time() - t0:.1f}s | nnz: {BtB.nnz:,}")
    n_items = neighbor_idx_matrix.shape[0]
    co_counts = np.zeros_like(neighbor_idx_matrix, dtype=np.int32)
    t1 = time.time()
    for item_idx in range(n_items):
        nbrs = neighbor_idx_matrix[item_idx]
        row_i = BtB.getrow(item_idx)
        co_counts[item_idx] = np.asarray(row_i[:, nbrs].todense()).ravel().astype(np.int32)
    print(f"  Index extraction done in {time.time() - t1:.1f}s")
    return co_counts

def predict_item_item_cf_fixed(df_subset, u_idx, i_idx, M_user_centered_csr, user_means_raw_vec, neighbor_idx_matrix, neighbor_sim_matrix, neighbor_common_matrix, user_mean_map_stage0, item_mean_map_stage0, global_mean, topk=50, min_common_users=2, min_neighbors_for_pred=3, shrinkage=10.0, min_sim_threshold=0.0):
    n = len(u_idx)
    preds = np.empty(n, dtype=np.float32)
    used_neighbor_counts = np.zeros(n, dtype=np.int32)
    fallback_flags = np.zeros(n, dtype=np.int8)
    raw_user_fallback = df_subset[COL_USER].map(user_mean_map_stage0).fillna(global_mean).astype(np.float32).values
    raw_item_fallback = df_subset[COL_ITEM].map(item_mean_map_stage0).fillna(global_mean).astype(np.float32).values
    baseline_fallback = 0.5 * (raw_user_fallback + raw_item_fallback)
    for t in range(n):
        u = u_idx[t]
        i = i_idx[t]
        row = M_user_centered_csr.getrow(u)
        rated_items = row.indices
        rated_vals = row.data
        if len(rated_items) == 0:
            preds[t] = baseline_fallback[t]
            fallback_flags[t] = 1
            continue
        nbr_idx = neighbor_idx_matrix[i, :topk]
        nbr_sim = neighbor_sim_matrix[i, :topk]
        nbr_cnt = neighbor_common_matrix[i, :topk]
        reliable_mask = (nbr_cnt >= min_common_users) & (nbr_sim > min_sim_threshold)
        nbr_idx = nbr_idx[reliable_mask]
        nbr_sim = nbr_sim[reliable_mask]
        nbr_cnt = nbr_cnt[reliable_mask]
        if len(nbr_idx) == 0:
            preds[t] = baseline_fallback[t]
            fallback_flags[t] = 1
            continue
        nbr_sim = nbr_sim * (nbr_cnt / (nbr_cnt + shrinkage))
        rated_map = dict(zip(rated_items, rated_vals))
        sims_used = []
        vals_used = []
        for item_j, sim_val in zip(nbr_idx, nbr_sim):
            if item_j in rated_map:
                sims_used.append(float(sim_val))
                vals_used.append(float(rated_map[item_j]))
        if len(sims_used) < min_neighbors_for_pred:
            preds[t] = baseline_fallback[t]
            fallback_flags[t] = 1
            continue
        sims_used = np.asarray(sims_used, dtype=np.float32)
        vals_used = np.asarray(vals_used, dtype=np.float32)
        denom = np.sum(np.abs(sims_used))
        if denom <= 1e-12:
            preds[t] = baseline_fallback[t]
            fallback_flags[t] = 1
            continue
        residual = np.sum(sims_used * vals_used) / denom
        preds[t] = np.clip(user_means_raw_vec[u] + residual, 1.0, 5.0)
        used_neighbor_counts[t] = len(sims_used)
    return preds.astype(np.float32), used_neighbor_counts, fallback_flags

item_neighbors_common_a_fit = compute_neighbor_co_counts(model_a_fit_artifacts["B"], item_neighbors_idx_a_fit)
print("item_neighbors_common_a_fit shape:", item_neighbors_common_a_fit.shape)
print("Example common counts             :", item_neighbors_common_a_fit[0][:10])


Computing B^T @ B (vectorized co-counts)...
  B^T @ B done in 0.2s | nnz: 9,425,117
  Index extraction done in 6.9s
item_neighbors_common_a_fit shape: (57261, 20)
Example common counts             : [1 1 1 1 1 1 1 1 1 1]


In [11]:
# ============================================
# CELL 10 ? TUNE MODEL A ON val_fit + SAVE OOF
# ============================================

print("=" * 70)
print("MODEL A ? TUNE ON val_fit")
print("=" * 70)

val_a_known_mask = val_fit[COL_USER].isin(user_enc_a_fit) & val_fit[COL_ITEM].isin(item_enc_a_fit)
result_base_a_oof = evaluate_predictions(y_val_true_oof, baseline_val_preds_oof, name="baseline_on_val_fit_model_a")
print_eval(result_base_a_oof)

topk_grid = [20]
min_common_grid = [1]
min_neighbors_grid = [1]
shrinkage_grid = [0.0, 2.0]
min_sim_grid = [0.0]
results_a_grid = []

for topk in topk_grid:
    for min_common in min_common_grid:
        for min_nbr in min_neighbors_grid:
            for shrink in shrinkage_grid:
                for min_sim in min_sim_grid:
                    preds_tmp = baseline_val_preds_oof.copy()
                    used_tmp_full = np.zeros(len(val_fit), dtype=np.int32)
                    fb_tmp_full = np.ones(len(val_fit), dtype=np.int8)
                    if val_a_known_mask.any():
                        val_known_df = val_fit.loc[val_a_known_mask].copy()
                        u_idx_tmp = val_known_df[COL_USER].map(user_enc_a_fit).astype(np.int32).values
                        i_idx_tmp = val_known_df[COL_ITEM].map(item_enc_a_fit).astype(np.int32).values
                        pred_known, used_known, fb_known = predict_item_item_cf_fixed(val_known_df, u_idx_tmp, i_idx_tmp, model_a_fit_artifacts["M_user_centered"], user_means_raw_a_fit, item_neighbors_idx_a_fit, item_neighbors_sim_a_fit, item_neighbors_common_a_fit, user_mean_map_fit, item_mean_map_fit, GLOBAL_MEAN_FIT, topk=topk, min_common_users=min_common, min_neighbors_for_pred=min_nbr, shrinkage=shrink, min_sim_threshold=min_sim)
                        preds_tmp[val_a_known_mask.to_numpy()] = pred_known
                        used_tmp_full[val_a_known_mask.to_numpy()] = used_known
                        fb_tmp_full[val_a_known_mask.to_numpy()] = fb_known
                    res_tmp = evaluate_predictions(y_val_true_oof, preds_tmp, name=f"model_a_topk{topk}_mc{min_common}_mn{min_nbr}_sh{shrink}_ms{min_sim}")
                    results_a_grid.append({"topk": int(topk), "min_common_users": int(min_common), "min_neighbors_for_pred": int(min_nbr), "shrinkage": float(shrink), "min_sim_threshold": float(min_sim), "rmse": float(res_tmp["rmse"]), "mae": float(res_tmp["mae"]), "n": int(res_tmp["n"]), "known_coverage": float(val_a_known_mask.mean()), "avg_used_neighbors": float(np.mean(used_tmp_full)), "median_used_neighbors": float(np.median(used_tmp_full)), "fallback_rate": float(np.mean(fb_tmp_full))})
                    print(f"topk={topk:<3d} | mc={min_common} | mn={min_nbr} | sh={shrink:<4.1f} | ms={min_sim:<4.2f} | OOF RMSE={res_tmp['rmse']:.6f} | OOF MAE={res_tmp['mae']:.6f}")

results_a_grid_df = pd.DataFrame(results_a_grid).sort_values(["rmse", "mae", "fallback_rate", "topk", "min_common_users", "min_neighbors_for_pred", "shrinkage", "min_sim_threshold"], ascending=[True, True, True, True, True, True, True, True]).reset_index(drop=True)
display(results_a_grid_df)

best_row_a = results_a_grid_df.iloc[0]
best_topk = int(best_row_a["topk"])
best_mc = int(best_row_a["min_common_users"])
best_mn = int(best_row_a["min_neighbors_for_pred"])
best_sh = float(best_row_a["shrinkage"])
best_ms = float(best_row_a["min_sim_threshold"])

model_a_val_preds_oof = baseline_val_preds_oof.copy()
model_a_val_used_neighbors_oof = np.zeros(len(val_fit), dtype=np.int32)
model_a_val_fallback_flags_oof = np.ones(len(val_fit), dtype=np.int8)
if val_a_known_mask.any():
    val_known_df = val_fit.loc[val_a_known_mask].copy()
    u_idx_val_a = val_known_df[COL_USER].map(user_enc_a_fit).astype(np.int32).values
    i_idx_val_a = val_known_df[COL_ITEM].map(item_enc_a_fit).astype(np.int32).values
    pred_known, used_known, fb_known = predict_item_item_cf_fixed(val_known_df, u_idx_val_a, i_idx_val_a, model_a_fit_artifacts["M_user_centered"], user_means_raw_a_fit, item_neighbors_idx_a_fit, item_neighbors_sim_a_fit, item_neighbors_common_a_fit, user_mean_map_fit, item_mean_map_fit, GLOBAL_MEAN_FIT, topk=best_topk, min_common_users=best_mc, min_neighbors_for_pred=best_mn, shrinkage=best_sh, min_sim_threshold=best_ms)
    model_a_val_preds_oof[val_a_known_mask.to_numpy()] = pred_known
    model_a_val_used_neighbors_oof[val_a_known_mask.to_numpy()] = used_known
    model_a_val_fallback_flags_oof[val_a_known_mask.to_numpy()] = fb_known

model_a_val_result_oof = evaluate_predictions(y_val_true_oof, model_a_val_preds_oof, name="model_a_val_oof")
print_eval(model_a_val_result_oof)
np.save(STAGE3_DIR / "model_a_val_preds_oof.npy", model_a_val_preds_oof)
np.save(STAGE3_DIR / "model_a_val_used_neighbors_oof.npy", model_a_val_used_neighbors_oof)
np.save(STAGE3_DIR / "model_a_val_fallback_flags_oof.npy", model_a_val_fallback_flags_oof)
results_a_grid_df.to_csv(STAGE3_DIR / "model_a_tuning.csv", index=False)


MODEL A ? TUNE ON val_fit
Model: baseline_on_val_fit_model_a
N    : 113117
RMSE : 1.046057
MAE  : 0.743151
------------------------------------------------------------
star_1_rmse   : 3.1409456729888916
star_1_mae    : 3.0944533348083496
star_1_n      : 5387
star_2_rmse   : 2.2216622829437256
star_2_mae    : 2.166825294494629
star_2_n      : 4077
star_3_rmse   : 1.3177480697631836
star_3_mae    : 1.2527437210083008
star_3_n      : 7411
star_4_rmse   : 0.5106449723243713
star_4_mae    : 0.4401628077030182
star_4_n      : 17070
star_5_rmse   : 0.6474452018737793
star_5_mae    : 0.5274767875671387
star_5_n      : 79172
topk=20  | mc=1 | mn=1 | sh=0.0  | ms=0.00 | OOF RMSE=1.048416 | OOF MAE=0.742965
topk=20  | mc=1 | mn=1 | sh=2.0  | ms=0.00 | OOF RMSE=1.048409 | OOF MAE=0.742963


,topk,min_common_users,min_neighbors_for_pred,shrinkage,min_sim_threshold,rmse,mae,n,known_coverage,avg_used_neighbors,median_used_neighbors,fallback_rate
0,20,1,1,2.0,0.0,1.048409,0.742963,113117,0.844126,0.010741,0.0,0.990399
1,20,1,1,0.0,0.0,1.048416,0.742965,113117,0.844126,0.010741,0.0,0.990399


Model: model_a_val_oof
N    : 113117
RMSE : 1.048409
MAE  : 0.742963
------------------------------------------------------------
star_1_rmse   : 3.141540050506592
star_1_mae    : 3.0916409492492676
star_1_n      : 5387
star_2_rmse   : 2.222156047821045
star_2_mae    : 2.1657679080963135
star_2_n      : 4077
star_3_rmse   : 1.3208093643188477
star_3_mae    : 1.2548601627349854
star_3_n      : 7411
star_4_rmse   : 0.5170644521713257
star_4_mae    : 0.44342681765556335
star_4_n      : 17070
star_5_rmse   : 0.6509054899215698
star_5_mae    : 0.5265527963638306
star_5_n      : 79172


In [12]:
# ============================================
# CELL 11 ? REFIT MODEL A ON FULL TRAIN + FINAL SAVE
# ============================================

print("=" * 70)
print("MODEL A ? REFIT BEST CONFIG ON FULL TRAIN")
print("=" * 70)
print("Best Model A config")
print("-" * 60)
print("topk                  :", best_topk)
print("min_common_users      :", best_mc)
print("min_neighbors_for_pred:", best_mn)
print("shrinkage             :", best_sh)
print("min_sim_threshold     :", best_ms)

model_a_full_artifacts = build_item_item_cf_artifacts(train)
user_enc_a = model_a_full_artifacts["user_enc"]
item_enc_a = model_a_full_artifacts["item_enc"]
user_means_raw_a = model_a_full_artifacts["user_means_raw"]
M_user_centered_a = model_a_full_artifacts["M_user_centered"]
item_neighbors_idx_a, item_neighbors_sim_a = precompute_item_neighbors(model_a_full_artifacts["M_user_centered"], max_k=MAX_K)
item_neighbors_common_a = compute_neighbor_co_counts(model_a_full_artifacts["B"], item_neighbors_idx_a)

known_mask_test_a = test_warm[COL_USER].isin(user_enc_a) & test_warm[COL_ITEM].isin(item_enc_a)
model_a_preds = y_pred_base.copy()
model_a_used_neighbors = np.zeros(len(test_warm), dtype=np.int32)
model_a_fallback_flags = np.ones(len(test_warm), dtype=np.int8)
if known_mask_test_a.any():
    test_known_df_a = test_warm.loc[known_mask_test_a].copy()
    u_idx_test_a = test_known_df_a[COL_USER].map(user_enc_a).astype(np.int32).values
    i_idx_test_a = test_known_df_a[COL_ITEM].map(item_enc_a).astype(np.int32).values
    pred_known, used_known, fb_known = predict_item_item_cf_fixed(test_known_df_a, u_idx_test_a, i_idx_test_a, M_user_centered_a, user_means_raw_a, item_neighbors_idx_a, item_neighbors_sim_a, item_neighbors_common_a, user_mean_map, item_mean_map, GLOBAL_MEAN, topk=best_topk, min_common_users=best_mc, min_neighbors_for_pred=best_mn, shrinkage=best_sh, min_sim_threshold=best_ms)
    model_a_preds[known_mask_test_a.to_numpy()] = pred_known
    model_a_used_neighbors[known_mask_test_a.to_numpy()] = used_known
    model_a_fallback_flags[known_mask_test_a.to_numpy()] = fb_known

assert len(model_a_preds) == len(test_warm), "Model A final predictions must align with test_warm."
result_a_best = evaluate_predictions(y_true_warm, model_a_preds, name="model_a_item_item_cf_full_train")
result_base_a = evaluate_predictions(y_true_warm, y_pred_base, name="baseline_on_test_warm")
print_eval(result_base_a)
print_eval(result_a_best)

rmse_gain_a = result_base_a["rmse"] - result_a_best["rmse"]
mae_gain_a = result_base_a["mae"] - result_a_best["mae"]
model_a_summary = {"best_topk": int(best_topk), "best_min_common_users": int(best_mc), "best_min_neighbors_for_pred": int(best_mn), "best_shrinkage": float(best_sh), "best_min_sim_threshold": float(best_ms), "val_fit_rmse": float(model_a_val_result_oof["rmse"]), "val_fit_mae": float(model_a_val_result_oof["mae"]), "val_fit_known_coverage": float(val_a_known_mask.mean()), "test_warm_rmse": float(result_a_best["rmse"]), "test_warm_mae": float(result_a_best["mae"]), "rmse_gain_vs_baseline_test_warm": float(rmse_gain_a), "mae_gain_vs_baseline_test_warm": float(mae_gain_a), "avg_used_neighbors": float(np.mean(model_a_used_neighbors)), "median_used_neighbors": float(np.median(model_a_used_neighbors)), "fallback_rate": float(np.mean(model_a_fallback_flags)), "test_known_coverage": float(known_mask_test_a.mean()), "test_warm_used_only_for_final_evaluation": True, "per_star": result_a_best["per_star"]}

np.save(STAGE3_DIR / "model_a_preds.npy", model_a_preds)
np.save(STAGE3_DIR / "model_a_used_neighbors.npy", model_a_used_neighbors)
np.save(STAGE3_DIR / "model_a_fallback_flags.npy", model_a_fallback_flags)
np.save(STAGE3_DIR / "model_a_neighbor_idx.npy", item_neighbors_idx_a)
np.save(STAGE3_DIR / "model_a_neighbor_sim.npy", item_neighbors_sim_a)
np.save(STAGE3_DIR / "model_a_neighbor_common.npy", item_neighbors_common_a)
with open(STAGE3_DIR / "model_a_result.json", "w") as f:
    json.dump(to_jsonable(result_a_best), f, indent=2)
with open(STAGE3_DIR / "model_a_summary.json", "w") as f:
    json.dump(to_jsonable(model_a_summary), f, indent=2)


MODEL A ? REFIT BEST CONFIG ON FULL TRAIN
Best Model A config
------------------------------------------------------------
topk                  : 20
min_common_users      : 1
min_neighbors_for_pred: 1
shrinkage             : 2.0
min_sim_threshold     : 0.0
Computing B^T @ B (vectorized co-counts)...
  B^T @ B done in 0.3s | nnz: 10,960,047
  Index extraction done in 7.0s
Model: baseline_on_test_warm
N    : 240285
RMSE : 1.102864
MAE  : 0.811304
------------------------------------------------------------
star_1_rmse   : 3.2606801986694336
star_1_mae    : 3.2519047260284424
star_1_n      : 14055
star_2_rmse   : 2.290545701980591
star_2_mae    : 2.280728340148926
star_2_n      : 9923
star_3_rmse   : 1.3291175365447998
star_3_mae    : 1.3141452074050903
star_3_n      : 16456
star_4_rmse   : 0.40190058946609497
star_4_mae    : 0.3692122995853424
star_4_n      : 36279
star_5_rmse   : 0.5842506885528564
star_5_mae    : 0.559917688369751
star_5_n      : 163572
Model: model_a_item_item_cf_ful

In [13]:
# ============================================
# CELL 12 ? MODEL B PREP (MF WITH SGD)
# ============================================
import numpy as np
import pandas as pd

print("=" * 70)
print("MODEL B ? MATRIX FACTORIZATION WITH SGD")
print("=" * 70)
assert "user_enc_a_fit" in globals(), "Run Model A tuning cells first."
assert "user_enc_a" in globals(), "Run Model A full-train refit cell first."

train_b_fit_encoded = train_fit.copy()
train_b_fit_encoded["user_idx"] = train_b_fit_encoded[COL_USER].map(user_enc_a_fit)
train_b_fit_encoded["item_idx"] = train_b_fit_encoded[COL_ITEM].map(item_enc_a_fit)
train_b_fit_encoded = train_b_fit_encoded.dropna(subset=["user_idx", "item_idx"]).copy()
train_b_fit_encoded["user_idx"] = train_b_fit_encoded["user_idx"].astype(np.int32)
train_b_fit_encoded["item_idx"] = train_b_fit_encoded["item_idx"].astype(np.int32)

u_train_fit_b = train_b_fit_encoded["user_idx"].values.astype(np.int32)
i_train_fit_b = train_b_fit_encoded["item_idx"].values.astype(np.int32)
r_train_fit_b = train_b_fit_encoded[COL_RATING].astype(np.float32).values
item_mean_fit_arr_b = train_b_fit_encoded[COL_ITEM].map(item_mean_map_fit).fillna(GLOBAL_MEAN_FIT).astype(np.float32).values
baseline_train_fit_b = 0.5 * (user_means_raw_a_fit[u_train_fit_b] + item_mean_fit_arr_b)
y_train_fit_b = (r_train_fit_b - baseline_train_fit_b).astype(np.float32)

val_b_known_mask = val_fit[COL_USER].isin(user_enc_a_fit) & val_fit[COL_ITEM].isin(item_enc_a_fit)

train_b_full_encoded = train.copy()
train_b_full_encoded["user_idx"] = train_b_full_encoded[COL_USER].map(user_enc_a)
train_b_full_encoded["item_idx"] = train_b_full_encoded[COL_ITEM].map(item_enc_a)
train_b_full_encoded = train_b_full_encoded.dropna(subset=["user_idx", "item_idx"]).copy()
train_b_full_encoded["user_idx"] = train_b_full_encoded["user_idx"].astype(np.int32)
train_b_full_encoded["item_idx"] = train_b_full_encoded["item_idx"].astype(np.int32)

u_train_b = train_b_full_encoded["user_idx"].values.astype(np.int32)
i_train_b = train_b_full_encoded["item_idx"].values.astype(np.int32)
r_train_b = train_b_full_encoded[COL_RATING].astype(np.float32).values
item_mean_train_arr_b_full = train_b_full_encoded[COL_ITEM].map(item_mean_map).fillna(GLOBAL_MEAN).astype(np.float32).values
baseline_train_b = 0.5 * (user_means_raw_a[u_train_b] + item_mean_train_arr_b_full)
y_train_b = (r_train_b - baseline_train_b).astype(np.float32)

known_mask_test_b = test_warm[COL_USER].isin(user_enc_a) & test_warm[COL_ITEM].isin(item_enc_a)
y_true_b = y_true_warm.copy()

print("Train-fit triplets:", len(train_b_fit_encoded))
print("Full-train triplets:", len(train_b_full_encoded))
print("Validation known coverage for Model B:", round(float(val_b_known_mask.mean()), 6))
print("Test known coverage for Model B:", round(float(known_mask_test_b.mean()), 6))
print("Residual target stats on train_fit:")
print(pd.Series(y_train_fit_b).describe())


MODEL B ? MATRIX FACTORIZATION WITH SGD
Train-fit triplets: 1018046
Full-train triplets: 1131163
Validation known coverage for Model B: 0.844126
Test known coverage for Model B: 1.0
Residual target stats on train_fit:
count    1.018046e+06
mean    -7.253358e-03
std      9.009053e-01
min     -5.613975e+00
25%     -2.804878e-01
50%      2.741935e-01
75%      5.208333e-01
max      2.769231e+00
dtype: float64


In [14]:
# ============================================
# CELL 13 — MODEL B CORE FUNCTIONS
# ============================================

def predict_mf_residual(u_idx, i_idx, P, Q):
    """
    residual prediction on user-centered target:
      residual_hat = dot(P[u], Q[i])
    """
    return np.sum(P[u_idx] * Q[i_idx], axis=1).astype(np.float32)


# ============================================
# CELL 13 — MODEL B CORE FUNCTIONS (FIXED)
# ============================================

def predict_model_b(
    df_subset,
    u_idx,
    i_idx,
    P, Q,
    bu, bi,
    user_means_raw_vec,
    item_mean_map,
    global_mean
):
    item_means = (
        df_subset[COL_ITEM].map(item_mean_map)
        .fillna(global_mean).astype(np.float32).values
    )
    baseline = 0.5 * (user_means_raw_vec[u_idx] + item_means)
    residual_hat = (
        bu[u_idx] + bi[i_idx]
        + np.sum(P[u_idx] * Q[i_idx], axis=1)
    ).astype(np.float32)
    preds = np.clip(baseline + residual_hat, 1.0, 5.0).astype(np.float32)
    return preds, residual_hat


from numba import njit, prange
 
@njit(parallel=False, cache=True)
def _sgd_epoch_numba(order, u_idx, i_idx, y_residual,
                     P, Q, bu, bi, lr, reg):
    """
    Single SGD epoch — compiled by Numba.
    parallel=False vì update P/Q có data dependency per user/item.
    cache=True để lần chạy sau không cần compile lại.
    
    Returns: sum of squared errors (để tính epoch RMSE)
    """
    se_sum = 0.0
    n_factors = P.shape[1]
    
    for idx in range(len(order)):
        t = order[idx]
        u = u_idx[t]
        i = i_idx[t]
        y = y_residual[t]
        
        # Compute prediction
        dot = 0.0
        for f in range(n_factors):
            dot += P[u, f] * Q[i, f]
        pred = bu[u] + bi[i] + dot
        
        err = y - pred
        se_sum += err * err
        
        # Update biases
        bu[u] += lr * (err - reg * bu[u])
        bi[i] += lr * (err - reg * bi[i])
        
        # Update factors
        for f in range(n_factors):
            pu_f = P[u, f]
            qi_f = Q[i, f]
            P[u, f] += lr * (err * qi_f - reg * pu_f)
            Q[i, f] += lr * (err * pu_f - reg * qi_f)
    
    return se_sum
 
 
def train_mf_sgd(                     # Tên giống hệt để không cần sửa Cell 15/16
    u_idx, i_idx, y_residual,
    n_users, n_items,
    n_factors=50,
    lr=0.01, reg=0.05,
    n_epochs=20,
    shuffle=True, random_state=42,
    verbose=True
):
    """
    SVD-style MF với Numba backend.
    Interface giống hệt version cũ — drop-in replacement.
    
    Lần đầu chạy sẽ mất ~30s để Numba compile.
    Các lần sau (cache=True) compile ngay lập tức.
    """
    rng = np.random.default_rng(random_state)
    
    P  = rng.normal(0, 0.01, (n_users, n_factors)).astype(np.float32)
    Q  = rng.normal(0, 0.01, (n_items, n_factors)).astype(np.float32)
    bu = np.zeros(n_users, dtype=np.float32)
    bi = np.zeros(n_items, dtype=np.float32)
    
    n = len(y_residual)
    order = np.arange(n, dtype=np.int32)
    
    # Warm-up compile: chạy 1 sample để trigger JIT
    if verbose:
        print("Numba JIT compiling (first run ~30s, cached after)...")
    _sgd_epoch_numba(
        order[:1],
        u_idx[:1].astype(np.int32),
        i_idx[:1].astype(np.int32),
        y_residual[:1].astype(np.float32),
        P, Q, bu, bi,
        np.float32(lr), np.float32(reg)
    )
    if verbose:
        print("Compiled. Starting training...")
    
    history = []
    cur_lr = np.float32(lr)
    
    for epoch in range(1, n_epochs + 1):
        if shuffle:
            rng.shuffle(order)
        
        se_sum = _sgd_epoch_numba(
            order,
            u_idx.astype(np.int32),
            i_idx.astype(np.int32),
            y_residual.astype(np.float32),
            P, Q, bu, bi,
            cur_lr, np.float32(reg)
        )
        
        epoch_rmse = float(np.sqrt(se_sum / n))
        cur_lr = np.float32(cur_lr * 0.95)
        history.append({"epoch": epoch, "train_residual_rmse": epoch_rmse})
        
        if verbose:
            print(f"Epoch {epoch:02d}/{n_epochs} | residual RMSE={epoch_rmse:.6f}")
    
    return P, Q, bu, bi, history

In [15]:
# ============================================
# CELL 14 ? MODEL B TRAIN_FIT / val_fit ARRAYS
# ============================================

u_fit_b = u_train_fit_b.astype(np.int32)
i_fit_b = i_train_fit_b.astype(np.int32)
y_fit_b = y_train_fit_b.astype(np.float32)

res_val_base = evaluate_predictions(y_val_true_oof, baseline_val_preds_oof, name="baseline_on_val_fit_model_b")
print_eval(res_val_base)
print("Train-fit size :", len(u_fit_b))
print("Val-fit size   :", len(val_fit))
print("Val date range :", date_range_str(val_fit))


Model: baseline_on_val_fit_model_b
N    : 113117
RMSE : 1.046057
MAE  : 0.743151
------------------------------------------------------------
star_1_rmse   : 3.1409456729888916
star_1_mae    : 3.0944533348083496
star_1_n      : 5387
star_2_rmse   : 2.2216622829437256
star_2_mae    : 2.166825294494629
star_2_n      : 4077
star_3_rmse   : 1.3177480697631836
star_3_mae    : 1.2527437210083008
star_3_n      : 7411
star_4_rmse   : 0.5106449723243713
star_4_mae    : 0.4401628077030182
star_4_n      : 17070
star_5_rmse   : 0.6474452018737793
star_5_mae    : 0.5274767875671387
star_5_n      : 79172
Train-fit size : 1018046
Val-fit size   : 113117
Val date range : 2017-01-04 00:00:00 -> 2017-05-06 00:00:00


In [16]:
# ============================================
# CELL 15 ? TUNE MODEL B ON val_fit + SAVE OOF
# ============================================

factor_grid = [20, 50]
lr_grid     = [0.01, 0.02]
reg_grid    = [0.02, 0.05]
epoch_grid  = [5, 10, 20, 30]

results_b_grid = []
best_model_b = None
best_model_b_key = (float("inf"), float("inf"))

for n_factors in factor_grid:
    for lr in lr_grid:
        for reg in reg_grid:
            for n_epochs in epoch_grid:
                P_tmp, Q_tmp, bu_tmp, bi_tmp, hist_tmp = train_mf_sgd(u_idx=u_fit_b, i_idx=i_fit_b, y_residual=y_fit_b, n_users=len(user_enc_a_fit), n_items=len(item_enc_a_fit), n_factors=n_factors, lr=lr, reg=reg, n_epochs=n_epochs, verbose=False)
                preds_tmp = baseline_val_preds_oof.copy()
                residual_tmp_full = np.zeros(len(val_fit), dtype=np.float32)
                if val_b_known_mask.any():
                    val_known_df_b = val_fit.loc[val_b_known_mask].copy()
                    u_val_b = val_known_df_b[COL_USER].map(user_enc_a_fit).astype(np.int32).values
                    i_val_b = val_known_df_b[COL_ITEM].map(item_enc_a_fit).astype(np.int32).values
                    pred_known, residual_known = predict_model_b(val_known_df_b, u_val_b, i_val_b, P_tmp, Q_tmp, bu_tmp, bi_tmp, user_means_raw_a_fit, item_mean_map_fit, GLOBAL_MEAN_FIT)
                    preds_tmp[val_b_known_mask.to_numpy()] = pred_known
                    residual_tmp_full[val_b_known_mask.to_numpy()] = residual_known
                res_val_tmp = evaluate_predictions(y_val_true_oof, preds_tmp, name=f"model_b_k{n_factors}_lr{lr}_reg{reg}_ep{n_epochs}")
                train_resid_rmse_last = hist_tmp[-1]["train_residual_rmse"]
                results_b_grid.append({"n_factors": int(n_factors), "lr": float(lr), "reg": float(reg), "n_epochs": int(n_epochs), "val_rmse": float(res_val_tmp["rmse"]), "val_mae": float(res_val_tmp["mae"]), "train_residual_rmse_last": float(train_resid_rmse_last), "known_coverage": float(val_b_known_mask.mean())})
                print(f"VAL RMSE={res_val_tmp['rmse']:.6f} | VAL MAE={res_val_tmp['mae']:.6f} | train_resid_RMSE={train_resid_rmse_last:.6f}")
                cur_key = (float(res_val_tmp["rmse"]), float(res_val_tmp["mae"]))
                if cur_key < best_model_b_key:
                    best_model_b_key = cur_key
                    best_model_b = {"P": P_tmp.copy(), "Q": Q_tmp.copy(), "bu": bu_tmp.copy(), "bi": bi_tmp.copy(), "history": hist_tmp, "config": {"n_factors": int(n_factors), "lr": float(lr), "reg": float(reg), "n_epochs": int(n_epochs)}, "val_result": res_val_tmp, "val_preds": preds_tmp.copy(), "val_residual": residual_tmp_full.copy()}

results_b_grid_df = pd.DataFrame(results_b_grid).sort_values(["val_rmse", "val_mae", "n_factors", "reg", "lr", "n_epochs"]).reset_index(drop=True)
display(results_b_grid_df)

best_cfg_b = best_model_b["config"]
model_b_val_preds_oof = best_model_b["val_preds"].astype(np.float32)
model_b_val_residual_oof = best_model_b["val_residual"].astype(np.float32)
model_b_val_result_oof = evaluate_predictions(y_val_true_oof, model_b_val_preds_oof, name="model_b_val_oof")
print("\nBest config on val_fit:")
print(best_cfg_b)
print("Best val RMSE:", model_b_val_result_oof["rmse"])
print("Baseline val RMSE:", res_val_base["rmse"])
print_eval(model_b_val_result_oof)

np.save(STAGE3_DIR / "model_b_val_preds_oof.npy", model_b_val_preds_oof)
np.save(STAGE3_DIR / "model_b_val_residual_oof.npy", model_b_val_residual_oof)
results_b_grid_df.to_csv(STAGE3_DIR / "model_b_tuning.csv", index=False)


VAL RMSE=1.046745 | VAL MAE=0.726074 | train_resid_RMSE=0.879415
VAL RMSE=1.050012 | VAL MAE=0.723836 | train_resid_RMSE=0.866711
VAL RMSE=1.054626 | VAL MAE=0.722944 | train_resid_RMSE=0.855075
VAL RMSE=1.057367 | VAL MAE=0.723152 | train_resid_RMSE=0.849128
VAL RMSE=1.046753 | VAL MAE=0.726269 | train_resid_RMSE=0.879537
VAL RMSE=1.049986 | VAL MAE=0.724060 | train_resid_RMSE=0.866987
VAL RMSE=1.054527 | VAL MAE=0.723157 | train_resid_RMSE=0.855785
VAL RMSE=1.057220 | VAL MAE=0.723348 | train_resid_RMSE=0.850716
VAL RMSE=1.051295 | VAL MAE=0.723630 | train_resid_RMSE=0.873226
VAL RMSE=1.058082 | VAL MAE=0.723395 | train_resid_RMSE=0.857931
VAL RMSE=1.066078 | VAL MAE=0.724687 | train_resid_RMSE=0.831279
VAL RMSE=1.070370 | VAL MAE=0.726071 | train_resid_RMSE=0.802500
VAL RMSE=1.051256 | VAL MAE=0.723851 | train_resid_RMSE=0.873470
VAL RMSE=1.057930 | VAL MAE=0.723584 | train_resid_RMSE=0.859055
VAL RMSE=1.065801 | VAL MAE=0.724819 | train_resid_RMSE=0.844028
VAL RMSE=1.069965 | VAL M

,n_factors,lr,reg,n_epochs,val_rmse,val_mae,train_residual_rmse_last,known_coverage
0,20,0.01,0.02,5,1.046745,0.726074,0.879415,0.844126
1,20,0.01,0.05,5,1.046753,0.726269,0.879537,0.844126
2,50,0.01,0.02,5,1.046773,0.726264,0.879222,0.844126
3,50,0.01,0.05,5,1.046782,0.726461,0.879365,0.844126
4,50,0.01,0.05,10,1.049980,0.724106,0.866597,0.844126
5,20,0.01,0.05,10,1.049986,0.724060,0.866987,0.844126
6,50,0.01,0.02,10,1.050007,0.723885,0.866224,0.844126
7,20,0.01,0.02,10,1.050012,0.723836,0.866711,0.844126
8,20,0.02,0.05,5,1.051256,0.723851,0.873470,0.844126
9,20,0.02,0.02,5,1.051295,0.723630,0.873226,0.844126



Best config on val_fit:
{'n_factors': 20, 'lr': 0.01, 'reg': 0.02, 'n_epochs': 5}
Best val RMSE: 1.0467449426651
Baseline val RMSE: 1.046057105064392
Model: model_b_val_oof
N    : 113117
RMSE : 1.046745
MAE  : 0.726074
------------------------------------------------------------
star_1_rmse   : 3.094911813735962
star_1_mae    : 3.030186891555786
star_1_n      : 5387
star_2_rmse   : 2.189859628677368
star_2_mae    : 2.1128249168395996
star_2_n      : 4077
star_3_rmse   : 1.3115878105163574
star_3_mae    : 1.2262541055679321
star_3_n      : 7411
star_4_rmse   : 0.552670419216156
star_4_mae    : 0.4702882766723633
star_4_n      : 17070
star_5_rmse   : 0.6632347106933594
star_5_mae    : 0.5062153935432434
star_5_n      : 79172


In [17]:
# ============================================
# CELL 16 ? REFIT BEST MODEL B ON FULL TRAIN
# ============================================

print("Refitting best Model B on full train...")
print(best_cfg_b)

P_b, Q_b, bu_b, bi_b, hist_b = train_mf_sgd(u_idx=u_train_b, i_idx=i_train_b, y_residual=y_train_b, n_users=len(user_enc_a), n_items=len(item_enc_a), n_factors=best_cfg_b["n_factors"], lr=best_cfg_b["lr"], reg=best_cfg_b["reg"], n_epochs=best_cfg_b["n_epochs"], shuffle=True, random_state=42, verbose=True)

model_b_preds = y_pred_base.copy()
model_b_residual = np.zeros(len(test_warm), dtype=np.float32)
if known_mask_test_b.any():
    test_known_df_b = test_warm.loc[known_mask_test_b].copy()
    u_idx_test_b = test_known_df_b[COL_USER].map(user_enc_a).astype(np.int32).values
    i_idx_test_b = test_known_df_b[COL_ITEM].map(item_enc_a).astype(np.int32).values
    pred_known, residual_known = predict_model_b(test_known_df_b, u_idx_test_b, i_idx_test_b, P_b, Q_b, bu_b, bi_b, user_means_raw_a, item_mean_map, GLOBAL_MEAN)
    model_b_preds[known_mask_test_b.to_numpy()] = pred_known
    model_b_residual[known_mask_test_b.to_numpy()] = residual_known

assert len(model_b_preds) == len(test_warm), "Model B final predictions must align with test_warm."
result_b = evaluate_predictions(y_true_warm, model_b_preds, name=(f"model_b_mf_sgd_k{best_cfg_b['n_factors']}_lr{best_cfg_b['lr']}_reg{best_cfg_b['reg']}_ep{best_cfg_b['n_epochs']}"))
result_base_b = evaluate_predictions(y_true_warm, y_pred_base, name="baseline_on_test_warm")
print_eval(result_base_b)
print_eval(result_b)
rmse_gain_b = result_base_b["rmse"] - result_b["rmse"]
mae_gain_b = result_base_b["mae"] - result_b["mae"]
print("RMSE gain over baseline test_warm:", rmse_gain_b)
print("MAE  gain over baseline test_warm:", mae_gain_b)
print("\nResidual stats on full test_warm:")
print(pd.Series(model_b_residual).describe())


Refitting best Model B on full train...
{'n_factors': 20, 'lr': 0.01, 'reg': 0.02, 'n_epochs': 5}
Numba JIT compiling (first run ~30s, cached after)...
Compiled. Starting training...
Epoch 01/5 | residual RMSE=0.923342
Epoch 02/5 | residual RMSE=0.911859
Epoch 03/5 | residual RMSE=0.903973
Epoch 04/5 | residual RMSE=0.897915
Epoch 05/5 | residual RMSE=0.893013
Model: baseline_on_test_warm
N    : 240285
RMSE : 1.102864
MAE  : 0.811304
------------------------------------------------------------
star_1_rmse   : 3.2606801986694336
star_1_mae    : 3.2519047260284424
star_1_n      : 14055
star_2_rmse   : 2.290545701980591
star_2_mae    : 2.280728340148926
star_2_n      : 9923
star_3_rmse   : 1.3291175365447998
star_3_mae    : 1.3141452074050903
star_3_n      : 16456
star_4_rmse   : 0.40190058946609497
star_4_mae    : 0.3692122995853424
star_4_n      : 36279
star_5_rmse   : 0.5842506885528564
star_5_mae    : 0.559917688369751
star_5_n      : 163572
Model: model_b_mf_sgd_k20_lr0.01_reg0.02_ep

In [18]:
# ============================================
# CELL 17 ? SAVE MODEL B
# ============================================

model_b_summary = {"best_config": to_jsonable(best_cfg_b), "val_fit_rmse": float(model_b_val_result_oof["rmse"]), "val_fit_mae": float(model_b_val_result_oof["mae"]), "val_fit_known_coverage": float(val_b_known_mask.mean()), "test_warm_rmse": float(result_b["rmse"]), "test_warm_mae": float(result_b["mae"]), "rmse_gain_vs_baseline_test_warm": float(rmse_gain_b), "mae_gain_vs_baseline_test_warm": float(mae_gain_b), "per_star": result_b["per_star"], "bu_abs_mean": float(np.abs(bu_b).mean()), "bi_abs_mean": float(np.abs(bi_b).mean()), "train_history": hist_b, "test_warm_used_only_for_final_evaluation": True}

np.save(STAGE3_DIR / "model_b_preds.npy", model_b_preds)
np.save(STAGE3_DIR / "model_b_residual.npy", model_b_residual)
np.save(STAGE3_DIR / "model_b_P.npy", P_b)
np.save(STAGE3_DIR / "model_b_Q.npy", Q_b)
np.save(STAGE3_DIR / "model_b_bu.npy", bu_b)
np.save(STAGE3_DIR / "model_b_bi.npy", bi_b)
with open(STAGE3_DIR / "model_b_result.json", "w") as f:
    json.dump(to_jsonable(result_b), f, indent=2)
with open(STAGE3_DIR / "model_b_summary.json", "w") as f:
    json.dump(to_jsonable(model_b_summary), f, indent=2)


In [19]:
# ============================================
# CELL 18 ? MODEL D PREP (ALS IMPLICIT)
# ============================================
"""
Model D uses implicit-feedback ALS.

Leakage rule here:
- Tune only on train_fit -> val_fit.
- Refit on full train only after best config is selected.
- test_warm is final evaluation only.
"""
import json
import warnings
warnings.filterwarnings("ignore")

print("=" * 70)
print("MODEL D ? ALS IMPLICIT FEEDBACK")
print("=" * 70)
print("Stage 2 C_implicit is loaded for reference only:")
print("  C_implicit shape:", C_implicit.shape)
print("  C_implicit nnz  :", C_implicit.nnz)
print("  train_fit rows  :", len(train_fit))
print("  val_fit rows    :", len(val_fit))

result_base_d = evaluate_predictions(y_true_warm, y_pred_base, name="baseline_on_test_warm_model_d")
print_eval(result_base_d)


MODEL D ? ALS IMPLICIT FEEDBACK
Stage 2 C_implicit is loaded for reference only:
  C_implicit shape: (164581, 58019)
  C_implicit nnz  : 1128092
  train_fit rows  : 1018046
  val_fit rows    : 113117
Model: baseline_on_test_warm_model_d
N    : 240285
RMSE : 1.102864
MAE  : 0.811304
------------------------------------------------------------
star_1_rmse   : 3.2606801986694336
star_1_mae    : 3.2519047260284424
star_1_n      : 14055
star_2_rmse   : 2.290545701980591
star_2_mae    : 2.280728340148926
star_2_n      : 9923
star_3_rmse   : 1.3291175365447998
star_3_mae    : 1.3141452074050903
star_3_n      : 16456
star_4_rmse   : 0.40190058946609497
star_4_mae    : 0.3692122995853424
star_4_n      : 36279
star_5_rmse   : 0.5842506885528564
star_5_mae    : 0.559917688369751
star_5_n      : 163572


In [20]:
# ============================================
# CELL 19 ? ALS CORE IMPLEMENTATION + MATRIX BUILDERS
# ============================================
import implicit

def als_implicit_train(C_csr, n_factors=20, n_iter=10, reg=0.01, alpha=1.0, random_state=42, verbose=True):
    C_scaled = C_csr.copy().astype(np.float64)
    if alpha != 1.0:
        C_scaled.data *= alpha
    model = implicit.als.AlternatingLeastSquares(factors=n_factors, iterations=n_iter, regularization=reg, random_state=random_state, use_gpu=False, calculate_training_loss=verbose)
    model.fit(C_scaled.tocsr(), show_progress=verbose)
    U = np.asarray(model.user_factors, dtype=np.float32)
    V = np.asarray(model.item_factors, dtype=np.float32)
    history = [{"iter": i + 1, "approx_loss": float("nan")} for i in range(n_iter)]
    return U, V, history

def build_implicit_artifacts(train_source_df):
    user_list = train_source_df[COL_USER].drop_duplicates().tolist()
    item_list = train_source_df[COL_ITEM].drop_duplicates().tolist()
    user_enc_local = {user_id: idx for idx, user_id in enumerate(user_list)}
    item_enc_local = {item_id: idx for idx, item_id in enumerate(item_list)}
    rows = train_source_df[COL_USER].map(user_enc_local).astype(np.int32).values
    cols = train_source_df[COL_ITEM].map(item_enc_local).astype(np.int32).values
    vals = np.ones(len(train_source_df), dtype=np.float32)
    C_local = sp.csr_matrix((vals, (rows, cols)), shape=(len(user_list), len(item_list)), dtype=np.float32)
    C_local.sum_duplicates()
    alpha_local = 40.0 if bool((C_local.data <= 1.0).all()) else 1.0
    return {"user_enc": user_enc_local, "item_enc": item_enc_local, "C": C_local, "alpha": float(alpha_local)}

print("ALS helpers ready.")


ALS helpers ready.


In [21]:
# ============================================
# CELL 20 ? MODEL D TRAIN_FIT / VAL_FIT ARTIFACTS
# ============================================

print("=" * 70)
print("MODEL D ? BUILD train_fit ARTIFACTS")
print("=" * 70)

model_d_fit_artifacts = build_implicit_artifacts(train_fit)
val_d_known_mask = val_fit[COL_USER].isin(model_d_fit_artifacts["user_enc"]) & val_fit[COL_ITEM].isin(model_d_fit_artifacts["item_enc"])

print("train_fit implicit shape:", model_d_fit_artifacts["C"].shape)
print("train_fit implicit nnz  :", model_d_fit_artifacts["C"].nnz)
print("alpha_fit              :", model_d_fit_artifacts["alpha"])
print("val_fit known coverage :", round(float(val_d_known_mask.mean()), 6))
print("val_fit date range     :", date_range_str(val_fit))

res_val_base_d = evaluate_predictions(y_val_true_oof, baseline_val_preds_oof, name="baseline_on_val_fit_model_d")
print_eval(res_val_base_d)


MODEL D ? BUILD train_fit ARTIFACTS
train_fit implicit shape: (160026, 57261)
train_fit implicit nnz  : 1015223
alpha_fit              : 1.0
val_fit known coverage : 0.844126
val_fit date range     : 2017-01-04 00:00:00 -> 2017-05-06 00:00:00
Model: baseline_on_val_fit_model_d
N    : 113117
RMSE : 1.046057
MAE  : 0.743151
------------------------------------------------------------
star_1_rmse   : 3.1409456729888916
star_1_mae    : 3.0944533348083496
star_1_n      : 5387
star_2_rmse   : 2.2216622829437256
star_2_mae    : 2.166825294494629
star_2_n      : 4077
star_3_rmse   : 1.3177480697631836
star_3_mae    : 1.2527437210083008
star_3_n      : 7411
star_4_rmse   : 0.5106449723243713
star_4_mae    : 0.4401628077030182
star_4_n      : 17070
star_5_rmse   : 0.6474452018737793
star_5_mae    : 0.5274767875671387
star_5_n      : 79172


In [22]:
# ============================================
# CELL 21 ? MODEL D FAST PREDICT HELPERS
# ============================================

def predict_als_rating_fast(baseline_arr, u_idx, i_idx, U, V, residual_scale=0.1, clip_lo=1.0, clip_hi=5.0):
    scores = np.sum(U[u_idx] * V[i_idx], axis=1).astype(np.float32)
    preds = np.clip(baseline_arr + residual_scale * scores, clip_lo, clip_hi).astype(np.float32)
    return preds, scores

def predict_als_with_fallback(df_subset, baseline_arr, implicit_artifacts, U, V, residual_scale):
    preds = baseline_arr.copy().astype(np.float32)
    scores = np.zeros(len(df_subset), dtype=np.float32)
    known_mask = df_subset[COL_USER].isin(implicit_artifacts["user_enc"]) & df_subset[COL_ITEM].isin(implicit_artifacts["item_enc"])
    if known_mask.any():
        known_df = df_subset.loc[known_mask].copy()
        u_idx = known_df[COL_USER].map(implicit_artifacts["user_enc"]).astype(np.int32).values
        i_idx = known_df[COL_ITEM].map(implicit_artifacts["item_enc"]).astype(np.int32).values
        pred_known, score_known = predict_als_rating_fast(baseline_arr[known_mask.to_numpy()], u_idx, i_idx, U, V, residual_scale=residual_scale)
        preds[known_mask.to_numpy()] = pred_known
        scores[known_mask.to_numpy()] = score_known
    return preds.astype(np.float32), scores.astype(np.float32), known_mask

print("Fast predict helper ready.")


Fast predict helper ready.


In [23]:
# ============================================
# CELL 22 ? MODEL D COARSE TUNING ON val_fit
# ============================================

print("=" * 70)
print("MODEL D ? COARSE TUNING ON val_fit")
print("=" * 70)

model_d_fallback_only = False
model_d_error_message = None
results_d_scale_df = pd.DataFrame()
best_rs_d_init = 0.0
U_d_init = None
V_d_init = None
hist_d_init = []
scale_grid_d = [-0.50, -0.25, -0.10, -0.05, 0.00, 0.01, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.50, 0.75, 1.00]

try:
    U_d_init, V_d_init, hist_d_init = als_implicit_train(model_d_fit_artifacts["C"].astype(np.float32), n_factors=20, n_iter=10, reg=0.01, alpha=model_d_fit_artifacts["alpha"], random_state=42, verbose=True)
    results_d_scale = []
    for rs in scale_grid_d:
        y_val_pred_tmp, scores_tmp, known_mask_tmp = predict_als_with_fallback(val_fit, baseline_val_preds_oof, model_d_fit_artifacts, U_d_init, V_d_init, rs)
        res_tmp = evaluate_predictions(y_val_true_oof, y_val_pred_tmp, name=f"d_val_rs_{rs}")
        results_d_scale.append({"residual_scale": float(rs), "rmse": float(res_tmp["rmse"]), "mae": float(res_tmp["mae"]), "known_coverage": float(known_mask_tmp.mean()), "score_mean": float(np.mean(scores_tmp)), "score_std": float(np.std(scores_tmp))})
        print(f"rs={rs:+.3f} | VAL RMSE={res_tmp['rmse']:.6f} | VAL MAE={res_tmp['mae']:.6f}")
    results_d_scale_df = pd.DataFrame(results_d_scale).sort_values(["rmse", "mae", "residual_scale"]).reset_index(drop=True)
    best_rs_d_init = float(results_d_scale_df.iloc[0]["residual_scale"])
except Exception as exc:
    model_d_fallback_only = True
    model_d_error_message = repr(exc)
    results_d_scale_df = pd.DataFrame([{"residual_scale": 0.0, "rmse": float(result_val_baseline_oof["rmse"]), "mae": float(result_val_baseline_oof["mae"]), "known_coverage": float(val_d_known_mask.mean()), "score_mean": 0.0, "score_std": 0.0}])
    print("Model D coarse tuning failed. Fallback to baseline OOF predictions.")
    print(model_d_error_message)

display(results_d_scale_df)
print("Best coarse residual_scale:", best_rs_d_init)


MODEL D ? COARSE TUNING ON val_fit


  0%|          | 0/10 [00:00<?, ?it/s]

rs=-0.500 | VAL RMSE=1.046091 | VAL MAE=0.743325
rs=-0.250 | VAL RMSE=1.046069 | VAL MAE=0.743236
rs=-0.100 | VAL RMSE=1.046061 | VAL MAE=0.743184
rs=-0.050 | VAL RMSE=1.046059 | VAL MAE=0.743168
rs=+0.000 | VAL RMSE=1.046057 | VAL MAE=0.743151
rs=+0.010 | VAL RMSE=1.046057 | VAL MAE=0.743148
rs=+0.030 | VAL RMSE=1.046056 | VAL MAE=0.743141
rs=+0.050 | VAL RMSE=1.046056 | VAL MAE=0.743135
rs=+0.080 | VAL RMSE=1.046055 | VAL MAE=0.743125
rs=+0.100 | VAL RMSE=1.046055 | VAL MAE=0.743118
rs=+0.150 | VAL RMSE=1.046055 | VAL MAE=0.743102
rs=+0.200 | VAL RMSE=1.046055 | VAL MAE=0.743086
rs=+0.300 | VAL RMSE=1.046056 | VAL MAE=0.743060
rs=+0.500 | VAL RMSE=1.046061 | VAL MAE=0.743016
rs=+0.750 | VAL RMSE=1.046070 | VAL MAE=0.742968
rs=+1.000 | VAL RMSE=1.046080 | VAL MAE=0.742923


,residual_scale,rmse,mae,known_coverage,score_mean,score_std
0,0.15,1.046055,0.743102,0.844126,0.000624,0.013175
1,0.20,1.046055,0.743086,0.844126,0.000624,0.013175
2,0.10,1.046055,0.743118,0.844126,0.000624,0.013175
3,0.08,1.046055,0.743125,0.844126,0.000624,0.013175
4,0.05,1.046056,0.743135,0.844126,0.000624,0.013175
5,0.30,1.046056,0.743060,0.844126,0.000624,0.013175
6,0.03,1.046056,0.743141,0.844126,0.000624,0.013175
7,0.01,1.046057,0.743148,0.844126,0.000624,0.013175
8,0.00,1.046057,0.743151,0.844126,0.000624,0.013175
9,-0.05,1.046059,0.743168,0.844126,0.000624,0.013175


Best coarse residual_scale: 0.15


In [24]:
# ============================================
# CELL 23 ? MODEL D FULL TUNING ON val_fit + SAVE OOF
# ============================================

print("=" * 70)
print("MODEL D ? FULL TUNING ON val_fit")
print("=" * 70)

factor_grid_d = [20, 50]
reg_grid_d = [0.005, 0.01]
iter_grid_d = [10, 15]
results_d_full = []
best_model_d = None
best_model_d_key = (float("inf"), float("inf"))

if not model_d_fallback_only:
    try:
        fine_scale_grid_d = sorted(set(list(np.round(np.linspace(0.8, 1.5, 15), 3)) + [0.0, float(best_rs_d_init)]))
        print("Fine scale grid:", fine_scale_grid_d)
        for n_factors in factor_grid_d:
            for reg in reg_grid_d:
                for n_iter in iter_grid_d:
                    print(f"\nTraining ALS: k={n_factors} | reg={reg} | iter={n_iter}")
                    U_tmp, V_tmp, hist_tmp = als_implicit_train(model_d_fit_artifacts["C"].astype(np.float32), n_factors=n_factors, n_iter=n_iter, reg=reg, alpha=model_d_fit_artifacts["alpha"], random_state=42, verbose=False)
                    best_rs_for_model = None
                    best_rmse_for_model = float("inf")
                    best_mae_for_model = float("inf")
                    best_pred_for_model = None
                    best_scores_for_model = None
                    for rs in fine_scale_grid_d:
                        y_val_pred_tmp, scores_tmp, _ = predict_als_with_fallback(val_fit, baseline_val_preds_oof, model_d_fit_artifacts, U_tmp, V_tmp, rs)
                        res_tmp = evaluate_predictions(y_val_true_oof, y_val_pred_tmp, name=f"d_val_k{n_factors}_reg{reg}_iter{n_iter}_rs{rs}")
                        cur_key = (float(res_tmp["rmse"]), float(res_tmp["mae"]))
                        if cur_key < (best_rmse_for_model, best_mae_for_model):
                            best_rmse_for_model = cur_key[0]
                            best_mae_for_model = cur_key[1]
                            best_rs_for_model = float(rs)
                            best_pred_for_model = y_val_pred_tmp.copy()
                            best_scores_for_model = scores_tmp.copy()
                    results_d_full.append({"n_factors": int(n_factors), "reg": float(reg), "n_iter": int(n_iter), "best_residual_scale": float(best_rs_for_model), "val_rmse": float(best_rmse_for_model), "val_mae": float(best_mae_for_model), "final_approx_loss": float(hist_tmp[-1]["approx_loss"])})
                    print(f"-> best_rs={best_rs_for_model:+.4f} | VAL RMSE={best_rmse_for_model:.6f} | VAL MAE={best_mae_for_model:.6f}")
                    cur_model_key = (float(best_rmse_for_model), float(best_mae_for_model))
                    if cur_model_key < best_model_d_key:
                        best_model_d_key = cur_model_key
                        best_model_d = {"U": U_tmp.copy(), "V": V_tmp.copy(), "history": hist_tmp, "config": {"n_factors": int(n_factors), "reg": float(reg), "n_iter": int(n_iter), "residual_scale": float(best_rs_for_model)}, "val_rmse": float(best_rmse_for_model), "val_mae": float(best_mae_for_model), "val_preds": best_pred_for_model, "val_scores": best_scores_for_model}
    except Exception as exc:
        model_d_fallback_only = True
        model_d_error_message = repr(exc)
        best_model_d = None
        print("Model D full tuning failed. Fallback to baseline OOF predictions.")
        print(model_d_error_message)

results_d_full_df = pd.DataFrame(results_d_full)
if len(results_d_full_df) > 0:
    results_d_full_df = results_d_full_df.sort_values(["val_rmse", "val_mae", "n_factors", "reg", "n_iter"]).reset_index(drop=True)
display(results_d_full_df)

if model_d_fallback_only or best_model_d is None:
    best_cfg_d = {"n_factors": None, "reg": None, "n_iter": None, "residual_scale": 0.0, "fallback_to_baseline": True, "error": model_d_error_message}
    model_d_val_preds_oof = baseline_val_preds_oof.copy()
    model_d_val_scores_oof = np.zeros(len(val_fit), dtype=np.float32)
else:
    best_cfg_d = best_model_d["config"]
    model_d_val_preds_oof = best_model_d["val_preds"].astype(np.float32)
    model_d_val_scores_oof = best_model_d["val_scores"].astype(np.float32)

model_d_val_result_oof = evaluate_predictions(y_val_true_oof, model_d_val_preds_oof, name="model_d_val_oof")
print_eval(model_d_val_result_oof)
np.save(STAGE3_DIR / "model_d_val_preds_oof.npy", model_d_val_preds_oof)
np.save(STAGE3_DIR / "model_d_val_scores_oof.npy", model_d_val_scores_oof)
results_d_full_df.to_csv(STAGE3_DIR / "model_d_tuning.csv", index=False)


MODEL D ? FULL TUNING ON val_fit
Fine scale grid: [0.0, 0.15, np.float64(0.8), np.float64(0.85), np.float64(0.9), np.float64(0.95), np.float64(1.0), np.float64(1.05), np.float64(1.1), np.float64(1.15), np.float64(1.2), np.float64(1.25), np.float64(1.3), np.float64(1.35), np.float64(1.4), np.float64(1.45), np.float64(1.5)]

Training ALS: k=20 | reg=0.005 | iter=10
-> best_rs=+0.1500 | VAL RMSE=1.046055 | VAL MAE=0.743102

Training ALS: k=20 | reg=0.005 | iter=15
-> best_rs=+0.1500 | VAL RMSE=1.046055 | VAL MAE=0.743102

Training ALS: k=20 | reg=0.01 | iter=10
-> best_rs=+0.1500 | VAL RMSE=1.046055 | VAL MAE=0.743102

Training ALS: k=20 | reg=0.01 | iter=15
-> best_rs=+0.1500 | VAL RMSE=1.046055 | VAL MAE=0.743102

Training ALS: k=50 | reg=0.005 | iter=10
-> best_rs=+0.1500 | VAL RMSE=1.046053 | VAL MAE=0.743094

Training ALS: k=50 | reg=0.005 | iter=15
-> best_rs=+0.1500 | VAL RMSE=1.046053 | VAL MAE=0.743093

Training ALS: k=50 | reg=0.01 | iter=10
-> best_rs=+0.1500 | VAL RMSE=1.04605

,n_factors,reg,n_iter,best_residual_scale,val_rmse,val_mae,final_approx_loss
0,50,0.005,15,0.15,1.046053,0.743093,NaN
1,50,0.010,15,0.15,1.046053,0.743093,NaN
2,50,0.005,10,0.15,1.046053,0.743094,NaN
3,50,0.010,10,0.15,1.046053,0.743094,NaN
4,20,0.005,10,0.15,1.046055,0.743102,NaN
5,20,0.010,10,0.15,1.046055,0.743102,NaN
6,20,0.005,15,0.15,1.046055,0.743102,NaN
7,20,0.010,15,0.15,1.046055,0.743102,NaN


Model: model_d_val_oof
N    : 113117
RMSE : 1.046053
MAE  : 0.743093
------------------------------------------------------------
star_1_rmse   : 3.1410093307495117
star_1_mae    : 3.0945186614990234
star_1_n      : 5387
star_2_rmse   : 2.2217774391174316
star_2_mae    : 2.1669366359710693
star_2_n      : 4077
star_3_rmse   : 1.3178175687789917
star_3_mae    : 1.2528144121170044
star_3_n      : 7411
star_4_rmse   : 0.5107342600822449
star_4_mae    : 0.4402696192264557
star_4_n      : 17070
star_5_rmse   : 0.6473656296730042
star_5_mae    : 0.5273541808128357
star_5_n      : 79172


In [25]:
# ============================================
# CELL 24 ? REFIT BEST MODEL D ON FULL TRAIN
# ============================================

print("=" * 70)
print("MODEL D ? REFIT BEST CONFIG ON FULL TRAIN")
print("=" * 70)
print(best_cfg_d)

model_d_full_artifacts = build_implicit_artifacts(train)
ALPHA_ALS_FULL = float(model_d_full_artifacts["alpha"])
U_d_best = np.zeros((1, 1), dtype=np.float32)
V_d_best = np.zeros((1, 1), dtype=np.float32)
hist_d_best = []
model_d_refit_status = "baseline_fallback"

if not bool(best_cfg_d.get("fallback_to_baseline", False)):
    try:
        U_d_best, V_d_best, hist_d_best = als_implicit_train(model_d_full_artifacts["C"].astype(np.float32), n_factors=best_cfg_d["n_factors"], n_iter=best_cfg_d["n_iter"], reg=best_cfg_d["reg"], alpha=ALPHA_ALS_FULL, random_state=42, verbose=True)
        model_d_refit_status = "trained"
    except Exception as exc:
        model_d_refit_status = "baseline_fallback"
        model_d_error_message = repr(exc)
        best_cfg_d["fallback_to_baseline"] = True
        best_cfg_d["full_refit_error"] = model_d_error_message
        print("Model D full refit failed. Falling back to baseline on test_warm.")
        print(model_d_error_message)


MODEL D ? REFIT BEST CONFIG ON FULL TRAIN
{'n_factors': 50, 'reg': 0.005, 'n_iter': 15, 'residual_scale': 0.15}


  0%|          | 0/15 [00:00<?, ?it/s]

In [26]:
# ============================================
# CELL 25 ? MODEL D FINAL TEST EVALUATION
# ============================================

print("=" * 70)
print("MODEL D ? FINAL TEST EVALUATION")
print("=" * 70)

if bool(best_cfg_d.get("fallback_to_baseline", False)):
    model_d_preds = y_pred_base.copy()
    scores_d_best = np.zeros(len(test_warm), dtype=np.float32)
    known_mask_test_d = np.zeros(len(test_warm), dtype=bool)
else:
    model_d_preds, scores_d_best, known_mask_test_d = predict_als_with_fallback(test_warm, y_pred_base, model_d_full_artifacts, U_d_best, V_d_best, float(best_cfg_d["residual_scale"]))

assert len(model_d_preds) == len(test_warm), "Model D final predictions must align with test_warm."
result_d_best = evaluate_predictions(y_true_warm, model_d_preds, name="model_d_als_full_train")
result_base_d_final = evaluate_predictions(y_true_warm, y_pred_base, name="baseline_on_test_warm")
print_eval(result_base_d_final)
print_eval(result_d_best)
rmse_gain_d = result_base_d_final["rmse"] - result_d_best["rmse"]
mae_gain_d = result_base_d_final["mae"] - result_d_best["mae"]
print("RMSE gain over baseline test_warm:", rmse_gain_d)
print("MAE  gain over baseline test_warm:", mae_gain_d)
print("\nALS score distribution:")
print(pd.Series(scores_d_best).describe())


MODEL D ? FINAL TEST EVALUATION
Model: baseline_on_test_warm
N    : 240285
RMSE : 1.102864
MAE  : 0.811304
------------------------------------------------------------
star_1_rmse   : 3.2606801986694336
star_1_mae    : 3.2519047260284424
star_1_n      : 14055
star_2_rmse   : 2.290545701980591
star_2_mae    : 2.280728340148926
star_2_n      : 9923
star_3_rmse   : 1.3291175365447998
star_3_mae    : 1.3141452074050903
star_3_n      : 16456
star_4_rmse   : 0.40190058946609497
star_4_mae    : 0.3692122995853424
star_4_n      : 36279
star_5_rmse   : 0.5842506885528564
star_5_mae    : 0.559917688369751
star_5_n      : 163572
Model: model_d_als_full_train
N    : 240285
RMSE : 1.102868
MAE  : 0.811267
------------------------------------------------------------
star_1_rmse   : 3.260756731033325
star_1_mae    : 3.251981496810913
star_1_n      : 14055
star_2_rmse   : 2.2906277179718018
star_2_mae    : 2.2808098793029785
star_2_n      : 9923
star_3_rmse   : 1.329219102859497
star_3_mae    : 1.3142

In [27]:
# ============================================
# CELL 26 ? MODEL D SAVE
# ============================================

print("=" * 70)
print("SAVING MODEL D ARTIFACTS")
print("=" * 70)

model_d_summary = {"best_config": to_jsonable(best_cfg_d), "alpha_fit": float(model_d_fit_artifacts["alpha"]), "alpha_full": float(ALPHA_ALS_FULL), "val_fit_rmse": float(model_d_val_result_oof["rmse"]), "val_fit_mae": float(model_d_val_result_oof["mae"]), "val_fit_known_coverage": float(val_d_known_mask.mean()), "test_warm_rmse": float(result_d_best["rmse"]), "test_warm_mae": float(result_d_best["mae"]), "rmse_gain_vs_baseline_test_warm": float(rmse_gain_d), "mae_gain_vs_baseline_test_warm": float(mae_gain_d), "test_known_coverage": float(known_mask_test_d.mean()), "per_star": result_d_best["per_star"], "train_history": hist_d_best, "refit_status": model_d_refit_status, "fallback_to_baseline": bool(best_cfg_d.get("fallback_to_baseline", False)), "error": model_d_error_message, "test_warm_used_only_for_final_evaluation": True}

np.save(STAGE3_DIR / "model_d_preds.npy", model_d_preds)
np.save(STAGE3_DIR / "model_d_scores.npy", scores_d_best)
np.save(STAGE3_DIR / "model_d_U.npy", U_d_best)
np.save(STAGE3_DIR / "model_d_V.npy", V_d_best)
with open(STAGE3_DIR / "model_d_result.json", "w") as f:
    json.dump(to_jsonable(result_d_best), f, indent=2)
with open(STAGE3_DIR / "model_d_summary.json", "w") as f:
    json.dump(to_jsonable(model_d_summary), f, indent=2)


SAVING MODEL D ARTIFACTS


In [28]:
# ============================================
# CELL 27 ? FINAL LENGTH CHECKS
# ============================================

final_test_pred_arrays = {"baseline_test": y_pred_base, "model_a_preds": model_a_preds, "model_b_preds": model_b_preds, "model_c_preds": model_c_preds, "model_d_preds": model_d_preds}
for arr_name, arr in final_test_pred_arrays.items():
    assert len(arr) == len(test_warm), f"Length mismatch for {arr_name}: {len(arr)} vs {len(test_warm)}"
    assert np.isfinite(arr).all(), f"Non-finite values found in {arr_name}"

print("All final test_warm prediction lengths are aligned:", len(test_warm))


All final test_warm prediction lengths are aligned: 240285


In [29]:
# ============================================
# CELL 28 - EXPORT STRICT OOF VALIDATION PREDICTIONS FOR STAGE 4
# ============================================

print('=' * 70)
print('EXPORT STRICT OOF VALIDATION PREDICTIONS FOR STAGE 4')
print('=' * 70)

oof_pred_arrays = {"y_val_true_oof": y_val_true_oof, "baseline_val_preds_oof": baseline_val_preds_oof, "model_a_val_preds_oof": model_a_val_preds_oof, "model_b_val_preds_oof": model_b_val_preds_oof, "model_c_val_preds_oof": model_c_val_preds_oof, "model_d_val_preds_oof": model_d_val_preds_oof}
for arr_name, arr in oof_pred_arrays.items():
    assert len(arr) == len(y_val_true_oof), f"Length mismatch for {arr_name}: {len(arr)} vs {len(y_val_true_oof)}"
    assert np.isfinite(arr).all(), f"Non-finite values found in {arr_name}"

validation_index_oof_df.to_csv(STAGE3_DIR / 'validation_index_oof.csv', index=False)
np.save(STAGE3_DIR / 'y_val_true_oof.npy', y_val_true_oof)
np.save(STAGE3_DIR / 'baseline_val_preds_oof.npy', baseline_val_preds_oof)
np.save(STAGE3_DIR / 'model_a_val_preds_oof.npy', model_a_val_preds_oof)
np.save(STAGE3_DIR / 'model_b_val_preds_oof.npy', model_b_val_preds_oof)
np.save(STAGE3_DIR / 'model_c_val_preds_oof.npy', model_c_val_preds_oof)
np.save(STAGE3_DIR / 'model_d_val_preds_oof.npy', model_d_val_preds_oof)

validation_preds_summary_oof = {'strict_oof': True, 'split_strategy': 'train sorted by review_date; train_fit = first 90%; val_fit = last 10%', 'n_train': int(len(train)), 'n_train_fit': int(len(train_fit)), 'n_val_fit': int(len(val_fit)), 'date_ranges': {'train_fit': date_range_str(train_fit), 'val_fit': date_range_str(val_fit), 'test_warm': date_range_str(test_warm)}, 'best_configs': {'model_a': {'topk': int(best_topk), 'min_common_users': int(best_mc), 'min_neighbors_for_pred': int(best_mn), 'shrinkage': float(best_sh), 'min_sim_threshold': float(best_ms)}, 'model_b': to_jsonable(best_cfg_b), 'model_c': {'residual_scale': float(best_rs)}, 'model_d': to_jsonable(best_cfg_d)}, 'metrics_on_val_fit': {'baseline': result_val_baseline_oof, 'model_a': model_a_val_result_oof, 'model_b': model_b_val_result_oof, 'model_c': model_c_val_result_oof, 'model_d': model_d_val_result_oof}, 'test_warm_policy': 'test_warm is used only for final evaluation, not tuning'}
with open(STAGE3_DIR / 'validation_preds_summary_oof.json', 'w') as f:
    json.dump(to_jsonable(validation_preds_summary_oof), f, indent=2)


EXPORT STRICT OOF VALIDATION PREDICTIONS FOR STAGE 4
